# Sistemas Especialistas

In [ ]:
## "Gambiarra" para compatibilidade da biblioteca Experta com Python 3.10+

import collections, collections.abc
collections.Mapping = collections.abc.Mapping
collections.MutableMapping = collections.abc.MutableMapping
collections.Sequence = collections.abc.Sequence
collections.Iterable = collections.abc.Iterable
collections.Callable = collections.abc.Callable

In [4]:
import experta

base_conhecimento = {
    "sol": {
        "quente": "Ir à Praia",
        "ameno": "Fazer uma caminhada no parque",
    },
    "nublado": {
        "quente": "Visitar um parente com ar condicionado",
        "ameno": "Ficar em casa e tomar café",
    },
    "chuva": {
        "quente": "Ficar na chuva",
        "ameno": "Ficar em casa comendo pipoca",
    }
}

## Motor de inferência (Simples)

In [5]:
def motor_inferencia_clima(fatos):
    clima = fatos.get("clima")
    temperatura = fatos.get("temperatura")
    
    if clima in base_conhecimento and temperatura in base_conhecimento[clima]:
        return base_conhecimento[clima][temperatura]
    else:
        return "Não tenho recomendação para essa combinação de clima e temperatura."

## Simulação

In [6]:
fatos = {
    "clima": "chuva",
    "temperatura": "quente"
}

conclusao = motor_inferencia_clima(fatos)
print(f"Fatos: {fatos}")
print(f"Conclusão: {conclusao}")

Fatos: {'clima': 'chuva', 'temperatura': 'quente'}
Conclusão: Ficar na chuva


# Forward e Backward Chaining

In [7]:
fatos = ["tem_pelos", "voa", "produz_leite"]
regras = [
    {
        "se": ["tem_penas", "voa"],
        "entao": "e_passaro"
    },
    {
        "se": ["tem_penas", "pode_cantar"],
        "entao": "e_canario"
    },
    {
        "se": ["tem_pelos", "produz_leite"],
        "entao": "e_mamifero"
    },
    {
        "se": ["tem_pelos", "voa"],
        "entao": "e_morcego"
    },
    {
        "se": ["tem_pelos", "pode_cantar"],
        "entao": "e_pirata"
    },
    {
        "se": ["tem_pelos", "late"],
        "entao": "e_cachorro"
    }
]

## Motor de inferência (Forward Chaining)

In [8]:
def motor_inferencia_forward(fatos_init, regras):
    fatos_derivados = list(fatos_init)
    novo_fato = True
    
    while novo_fato:
        novo_fato = False
        for regra in regras:
            condicao_satisfeita = all(condicao in fatos_derivados for condicao in regra["se"])
            
            if condicao_satisfeita and regra["entao"] not in fatos_derivados:
                fatos_derivados.append(regra["entao"])
                print(f"Regra disparada: SE {regra['se']} ENTAO {regra['entao']}")
                print(f"Fato adicionado: {regra['entao']}")
                novo_fato = True
                
    return fatos_derivados           

## Simulação (Forward Chaining)

In [9]:
print(f"Fatos iniciais: {fatos}")
fatos_finais = motor_inferencia_forward(fatos, regras)
print(f"Fatos finais: {fatos_finais}")

Fatos iniciais: ['tem_pelos', 'voa', 'produz_leite']
Regra disparada: SE ['tem_pelos', 'produz_leite'] ENTAO e_mamifero
Fato adicionado: e_mamifero
Regra disparada: SE ['tem_pelos', 'voa'] ENTAO e_morcego
Fato adicionado: e_morcego
Fatos finais: ['tem_pelos', 'voa', 'produz_leite', 'e_mamifero', 'e_morcego']


## Biblioteca Experta

In [10]:
from experta import *

class Caracteristica(Fact):
    "Representa uma característica observada"
    pass

class Animal(KnowledgeEngine):
    @DefFacts()
    def fatos_iniciais(self):
        yield Caracteristica("tem_penas")
        yield Caracteristica("voa")
        yield Caracteristica("pode_cantar")
        print("Fatos iniciais carregados.")

    @Rule(Caracteristica("tem_penas"), Caracteristica("voa"))
    def regra_e_passaro(self):
        print("Regra disparada: SE tem_penas e voa ENTAO e_passaro")
        self.declare(Fact(animal="e_passaro"))

    @Rule(Fact(animal="e_passaro"), Caracteristica("pode_cantar"))
    def regra_e_canario(self):
        print("Regra disparada: SE e_passaro e pode_cantar ENTAO e_canario")
        self.declare(Fact(animal="canário"))

    @Rule(Fact(animal=MATCH.tipo))
    def print_resultado(self, tipo):
        if tipo == "canário":
            print(f"Conclusão final do sistema, o tipo de anial é: {tipo}")

animal = Animal()
animal.reset()
animal.run()

Fatos iniciais carregados.
Regra disparada: SE tem_penas e voa ENTAO e_passaro
Regra disparada: SE e_passaro e pode_cantar ENTAO e_canario
Conclusão final do sistema, o tipo de anial é: canário
